# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amey05081999/Flyrank-Internship-Amey-Naik/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)


This notebook builds one transparent, label-free baseline for a ranked content-review queue.

**Rule idea:** prioritize pages that are both visibly stale and/or show a low-CTR opportunity at a strong search position. Visibility is used as the evidence floor, not as a future outcome.

The two signal checks below are deliberately skeptical. A negative verdict is useful because it prevents a weak assumption from becoming a rule input.

## 1. Signal checks

### Signal 1 — Staleness behind the refresh flag

**Claim:** older update age should identify pages with meaningful current search visibility.

The existing FlyRank refresh logic uses `days_since_last_update >= 180` together with `impressions_90d >= 500` for `stale_visible_page`. I test the staleness part without using any trend/label field.

**Verdict: OPPOSITE.** In this starter slice, pages at least 181 days since update have much lower current visibility than pages updated within 180 days. The result says **staleness alone is not a reason to rank a page highly**; the visibility gate is doing important work.

In [4]:
import requests
from pathlib import Path

# URL of the raw CSV file on GitHub
csv_url = "https://raw.githubusercontent.com/amey05081999/Flyrank-Internship-Amey-Naik/main/data/raw/content_refresh_anonymized.csv"

# Define the local path where the file should be saved
# This assumes Colab's default working directory is /content
local_dir = Path("/content/data/raw")
local_dir.mkdir(parents=True, exist_ok=True) # Create the directory if it doesn't exist
local_file_path = local_dir / "content_refresh_anonymized.csv"

# Download the file
response = requests.get(csv_url)
if response.status_code == 200:
    with open(local_file_path, "wb") as f:
        f.write(response.content)
    print(f"File downloaded successfully to {local_file_path}")
else:
    print(f"Failed to download file. Status code: {response.status_code}")

File downloaded successfully to /content/data/raw/content_refresh_anonymized.csv


In [5]:
from pathlib import Path
import numpy as np
import pandas as pd

# The file was downloaded to /content/data/raw/content_refresh_anonymized.csv.
# Update DATA_PATH to point directly to this location.
DATA_PATH = Path("/content/data/raw/content_refresh_anonymized.csv")

# Also explicitly define OUT_DIR for consistency with Colab's typical filesystem structure.
OUT_DIR = Path("/content/work/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f"Rows: {len(df):,}; columns: {len(df.columns)}")

df["stale_group"] = np.where(
    df["days_since_last_update"] >= 180,
    "181+ days",
    "<180 days",
)

staleness_table = (
    df.groupby("stale_group", sort=False)
      .agg(
          n=("content_id", "size"),
          median_impressions_90d=("impressions_90d", "median"),
          visible_500_pct=("impressions_90d", lambda s: 100 * (s >= 500).mean()),
          median_ctr=("ctr", "median"),
      )
      .reset_index()
)

display(staleness_table)
assert (staleness_table["n"] >= 50).all(), "Signal verdicts need at least 50 rows per bucket."

Rows: 30,000; columns: 44


,stale_group,n,median_impressions_90d,visible_500_pct,median_ctr
0,<180 days,29826,742.0,56.021592,0.07
1,181+ days,174,15.5,9.770115,0.00


### Signal 2 — CTR versus position behind the CTR-fix logic

**Claim:** a low CTR is more useful as an opportunity signal when a page already has a strong search position.

This is linked to FlyRank's `low_ctr_visible_page` logic: at least 500 impressions, position 1–20, and CTR below 0.5. I use only current-window fields.

**Verdict: MIXED.** Pages at positions 1–10 have a higher median CTR than pages at 11–20 (0.24% vs 0.17%), supporting the idea that strong position gives useful context. But low CTR is still very common even at positions 1–10 (78.9%), so CTR alone is not a precise diagnosis.

In [6]:
ctr_frame = df[
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
].copy()

ctr_frame["position_bucket"] = np.where(
    ctr_frame["avg_position"] <= 10,
    "1-10",
    "11-20",
)

ctr_table = (
    ctr_frame.groupby("position_bucket", sort=False)
             .agg(
                 n=("content_id", "size"),
                 median_ctr=("ctr", "median"),
                 mean_ctr=("ctr", "mean"),
                 low_ctr_pct=("ctr", lambda s: 100 * (s < 0.5).mean()),
                 median_impressions_90d=("impressions_90d", "median"),
             )
             .reset_index()
)

display(ctr_table)
assert (ctr_table["n"] >= 50).all(), "Signal verdicts need at least 50 rows per bucket."



,position_bucket,n,median_ctr,mean_ctr,low_ctr_pct,median_impressions_90d
0,11-20,4459,0.17,0.266629,84.996636,2166.0
1,1-10,7564,0.24,0.338872,78.913273,4484.0


## 2. One transparent rule and ranked queue

**Plain-language rule:** rank pages higher when they have substantial current visibility and either (a) have gone 180+ days without an update or (b) have CTR below 0.5% while ranking in positions 1–20. A page satisfying both conditions gets the strongest review priority.

The score has no fitted weights and uses no label-derived or future-window fields:

- `visibility_score = percentile rank of log1p(impressions_90d)`
- add 1 point for the stale-visible condition
- add 1 point for the CTR-opportunity condition

Each row receives **exactly one reason code** and **one action label**.

In [7]:
# Label/future-window guard: these fields must never enter the rule.
FORBIDDEN = {
    "trend_direction", "trend_pct", "is_declining_label",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
}
RULE_FIELDS = {
    "days_since_last_update", "impressions_90d", "avg_position", "ctr"
}
assert RULE_FIELDS.isdisjoint(FORBIDDEN)

q = df.copy()

# Heavy-tailed traffic is rank-transformed after log1p, following the signal-audit guidance.
log_visibility = np.log1p(q["impressions_90d"].clip(lower=0))
visibility_score = log_visibility.rank(method="average", pct=True)

q["stale_visible"] = (
    (q["days_since_last_update"] >= 180)
    & (q["impressions_90d"] >= 500)
)

q["ctr_opportunity"] = (
    (q["impressions_90d"] >= 500)
    & (q["avg_position"] > 0)
    & (q["avg_position"] <= 20)
    & (q["ctr"] < 0.5)
)

q["baseline_action_score"] = (
    visibility_score
    + q["stale_visible"].astype(int)
    + q["ctr_opportunity"].astype(int)
)

q["reason_code"] = np.select(
    [
        q["stale_visible"] & q["ctr_opportunity"],
        q["stale_visible"],
        q["ctr_opportunity"],
    ],
    [
        "stale_and_ctr",
        "stale_visible",
        "ctr_opportunity",
    ],
    default="monitor",
)

q["action"] = np.select(
    [
        q["stale_visible"] & q["ctr_opportunity"],
        q["stale_visible"],
        q["ctr_opportunity"],
    ],
    [
        "refresh_and_review_ctr",
        "refresh",
        "review_ctr",
    ],
    default="monitor",
)

q = q.sort_values(
    ["baseline_action_score", "impressions_90d", "content_id"],
    ascending=[False, False, True],
).reset_index(drop=True)

q["baseline_rank"] = np.arange(1, len(q) + 1)

queue_columns = [
    "baseline_rank",
    "content_id",
    "baseline_action_score",
    "reason_code",
    "action",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr",
]

queue = q[queue_columns].copy()
queue_path = OUT_DIR / "baseline_action_score.csv"
queue.to_csv(queue_path, index=False)

print(f"Wrote ranked queue: {queue_path}")
print(f"Rows ranked: {len(queue):,}")
display(queue.head(10))


Wrote ranked queue: /content/work/outputs/baseline_action_score.csv
Rows ranked: 30,000


,baseline_rank,content_id,baseline_action_score,reason_code,action,impressions_90d,days_since_last_update,avg_position,ctr
0,1,content_cf56e2e2e282,2.986600,stale_and_ctr,refresh_and_review_ctr,61678,194,19.7,0.15
1,2,content_0a91db491d14,2.908100,stale_and_ctr,refresh_and_review_ctr,13299,193,10.5,0.49
2,3,content_c2d929d83eaa,2.848867,stale_and_ctr,refresh_and_review_ctr,7558,193,17.9,0.20
3,4,content_fe16a55cd13d,2.782850,stale_and_ctr,refresh_and_review_ctr,4556,194,16.4,0.33
4,5,content_928af3e22c80,2.632783,stale_and_ctr,refresh_and_review_ctr,1697,193,15.8,0.12
5,6,content_e3ff1b093148,2.603667,stale_and_ctr,refresh_and_review_ctr,1408,183,7.8,0.28
6,7,content_7f116ae1f6f5,2.542400,stale_and_ctr,refresh_and_review_ctr,954,301,9.0,0.42
7,8,content_77d4d5930e5e,2.519983,stale_and_ctr,refresh_and_review_ctr,828,194,18.6,0.24
8,9,content_72496874f806,2.518833,stale_and_ctr,refresh_and_review_ctr,821,301,5.8,0.24
9,10,content_6226ee6adc91,2.455583,stale_and_ctr,refresh_and_review_ctr,545,183,17.8,0.18


## 3. Top-10 review

The review is deliberately concrete: each row states the action, why the rule ranked it, and what evidence would make the recommendation wrong.

In [8]:
top10 = q.head(10).copy()
review_rows = []

for _, r in top10.iterrows():
    if r["reason_code"] == "stale_and_ctr":
        why = (
            f"stale ({int(r['days_since_last_update'])} days) + low CTR ({r['ctr']:.2f}%) "
            f"with position {r['avg_position']:.1f} and {int(r['impressions_90d']):,} impressions"
        )
        wrong = "Wrong if the page was intentionally left unchanged, the query mix is irrelevant, or CTR is low for a known SERP feature."
    elif r["reason_code"] == "ctr_opportunity":
        why = (
            f"CTR opportunity ({r['ctr']:.2f}%) at position {r['avg_position']:.1f} "
            f"with {int(r['impressions_90d']):,} impressions"
        )
        wrong = "Wrong if the apparent CTR gap is caused by SERP layout, query intent, or tracking rather than content quality."
    else:
        why = (
            f"stale visible page ({int(r['days_since_last_update'])} days; "
            f"{int(r['impressions_90d']):,} impressions)"
        )
        wrong = "Wrong if the page is intentionally evergreen and its current traffic is healthy despite the age."
    review_rows.append({
        "rank": int(r["baseline_rank"]),
        "action": r["action"],
        "why": why,
        "what_would_make_it_wrong": wrong,
    })

review = pd.DataFrame(review_rows)
display(review)



,rank,action,why,what_would_make_it_wrong
0,1,refresh_and_review_ctr,stale (194 days) + low CTR (0.15%) with positi...,Wrong if the page was intentionally left uncha...
1,2,refresh_and_review_ctr,stale (193 days) + low CTR (0.49%) with positi...,Wrong if the page was intentionally left uncha...
2,3,refresh_and_review_ctr,stale (193 days) + low CTR (0.20%) with positi...,Wrong if the page was intentionally left uncha...
3,4,refresh_and_review_ctr,stale (194 days) + low CTR (0.33%) with positi...,Wrong if the page was intentionally left uncha...
4,5,refresh_and_review_ctr,stale (193 days) + low CTR (0.12%) with positi...,Wrong if the page was intentionally left uncha...
5,6,refresh_and_review_ctr,stale (183 days) + low CTR (0.28%) with positi...,Wrong if the page was intentionally left uncha...
6,7,refresh_and_review_ctr,stale (301 days) + low CTR (0.42%) with positi...,Wrong if the page was intentionally left uncha...
7,8,refresh_and_review_ctr,stale (194 days) + low CTR (0.24%) with positi...,Wrong if the page was intentionally left uncha...
8,9,refresh_and_review_ctr,stale (301 days) + low CTR (0.24%) with positi...,Wrong if the page was intentionally left uncha...
9,10,refresh_and_review_ctr,stale (183 days) + low CTR (0.18%) with positi...,Wrong if the page was intentionally left uncha...


## 4. Weak picks

Two weaknesses are visible in the rule:

1. **Staleness is a weak signal by itself.** The signal test was OPPOSITE: 181+ day pages had median 15.5 impressions versus 742 for <180-day pages. The rule therefore requires visibility before staleness can add score.
2. **Low CTR is not a diagnosis.** Even positions 1–10 had 78.9% of pages below 0.5% CTR in the tested slice. A human should check SERP features, query mix, and measurement before changing content.

The top list can therefore contain false positives, especially pages whose low CTR has a non-content explanation. That is an intended limitation of a transparent baseline, not a reason to hide the weakness.

In [9]:
# A compact leakage/self-check that is visible in the executed notebook.
assert "trend_direction" not in RULE_FIELDS
assert "trend_pct" not in RULE_FIELDS
assert "is_declining_label" not in RULE_FIELDS
assert not any(f.endswith("_last_30d") or f.endswith("_prev_30d") for f in RULE_FIELDS)

# Exactly one reason code and action label per row.
assert queue["reason_code"].notna().all()
assert queue["action"].notna().all()

print("Weak-pick / leakage checks:")
print("- No trend_direction, trend_pct, or is_declining_label in rule fields.")
print("- No last-30d or previous-30d window fields in rule fields.")
print("- One reason_code and one action per ranked row.")
print("- The negative staleness verdict was retained as a guardrail: stale pages must also be visible.")



Weak-pick / leakage checks:
- No trend_direction, trend_pct, or is_declining_label in rule fields.
- No last-30d or previous-30d window fields in rule fields.
- One reason_code and one action per ranked row.
- The negative staleness verdict was retained as a guardrail: stale pages must also be visible.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.